In [13]:
pip install --upgrade ultralytics

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 386.5 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.5 MB 386.5 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.5 MB 386.5 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.5 MB 386.5 kB/s eta 0:00:03
   -------------- ------------------------- 0.5/1.5 MB 386.5 kB/s eta 0:00:03
   --------------------- ------------------ 0.8/1.5 MB 310.2 kB/s eta 0:00:03
   --------------------- ------------------ 0.8/1.5 MB 310.2 kB/s eta 0:00:03
   --------------------- ---------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from ultralytics import YOLO
import torch

print("YOLO imported successfully!")
print("PyTorch version:", torch.__version__)

# Check and set up GPU
if torch.cuda.is_available():
    torch.cuda.set_device(0)
    torch.cuda.empty_cache()
    print(f" GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("CUDA not available - falling back to CPU")
    print("   Install PyTorch with CUDA: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
    device = 'cpu'

print(f"Using device: {device}")

YOLO imported successfully!
PyTorch version: 2.7.1+cu118
 GPU detected: NVIDIA GeForce RTX 2060
   VRAM: 6.4 GB
Using device: cuda


In [15]:
# train_RGB.ipynb
# Cell 1 — imports and sanity check

from ultralytics import YOLO
import torch, os

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version : 2.7.1+cu118
CUDA available  : True
GPU             : NVIDIA GeForce RTX 2060
VRAM            : 6.4 GB


In [16]:
import os

# Change to Desktop
os.chdir('C:/Users/DELL/Desktop/c')

# Verify
print("Current working directory:", os.getcwd())
print("\nContents of Desktop:")
print(os.listdir('.'))

Current working directory: C:\Users\DELL\Desktop\c

Contents of Desktop:
['Codes', 'conv.m', 'conversion.m', 'rgb_passthrough.m', 'rgb_to_lgb.m', 'rgb_to_lgbv2.m', 'rgb_to_oklab_full.m', 'rgb_to_ygb.m', 'S3_verify_conversion.m', 'S4_build_datasets.m', 'SIMD_LGB', 'SIMD_RGB', 'SIMD_YGB']


In [17]:
import os

# Check what's in your SIMD_RGB directory
dataset_path = 'SIMD_RGB'
if os.path.exists(dataset_path):
    print("Contents of SIMD_RGB:")
    for item in os.listdir(dataset_path):
        print(f"  - {item}")
        
    # Check if train/val directories exist
    for folder in ['train', 'val', 'test']:
        folder_path = os.path.join(dataset_path, folder)
        if os.path.exists(folder_path):
            print(f"\n{folder} contents:")
            for subitem in os.listdir(folder_path):
                print(f"  - {subitem}")
else:
    print(f"SIMD_RGB directory not found at {dataset_path}")
    print(f"Current location: {os.getcwd()}")

Contents of SIMD_RGB:
  - data.yaml
  - test
  - train
  - val

train contents:
  - images
  - labels

val contents:
  - images
  - labels

test contents:
  - images
  - labels


In [18]:
# Run this once before training
import os

cache_files = [
    'SIMD_RGB/train/labels.cache',
    'SIMD_RGB/val/labels.cache',
    'SIMD_RGB/train/labels.cache.npy',
    'SIMD_RGB/val/labels.cache.npy'
]

for cf in cache_files:
    if os.path.exists(cf):
        os.remove(cf)
        print(f"Removed: {cf}")
        
print("Cache cleared. Restart training and warnings should disappear.")

Cache cleared. Restart training and warnings should disappear.


In [19]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU device: {torch.cuda.get_device_name(0)}")
print(f"Current device: {torch.cuda.current_device()}")

# Force GPU usage
torch.cuda.set_device(0)

CUDA available: True
GPU device: NVIDIA GeForce RTX 2060
Current device: 0


## RGB  YOLOv8

In [20]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
N_total      = 35577
C            = 15

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [15]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolov8n.pt')    
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SIMD_RGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SIMD_RGB_YOLOv8n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.162 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_RGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt

In [21]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_RGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_RGB_YOLOv8n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_RGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_RGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON RGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB VALIDATION RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON RGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB TEST RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('RGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("RGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")

print("\n✓ Results saved to 'RGB_COMPLETE_results_nc15.txt'")
print("\n" + "="*50)
print("RGB COMPLETE!")
print("="*50)

Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_RGB\test\labels.cache
✓ Created: SIMD_RGB_val_nc15.yaml
✓ Created: SIMD_RGB_test_nc15.yaml

RUNNING VALIDATION ON RGB
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 408.698.7 MB/s, size: 263.6 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 1.9Kit/s 0.3s<0.1s
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.9it/s 6.5s0.2s
                   all        500       3218      0.729      0.722      0.758      0.595
                   car        230        909

## YGB  YOLOv8

In [24]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
N_total      = 35577
C            = 15

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [15]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolov8n.pt')    
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SIMD_YGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SIMD_YGB_YOLOv8n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.162 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_YGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt

In [25]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_YGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_YGB_YOLOv8n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_YGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_YGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON YGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB VALIDATION RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON YGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB TEST RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('YGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("YGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")

print("\n✓ Results saved to 'YGB_COMPLETE_results_nc15.txt'")
print("\n" + "="*50)
print("YGB COMPLETE!")
print("="*50)

Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_YGB\test\labels.cache
✓ Created: SIMD_YGB_val_nc15.yaml
✓ Created: SIMD_YGB_test_nc15.yaml

RUNNING VALIDATION ON YGB
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 426.970.6 MB/s, size: 241.1 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels... 759 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 2.1Kit/s 0.4s0.1ss
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 4.9it/s 9.7s0.2s
                   all        759       6720      0.713      0.737      0.778      0.614
                   car        383       2544

## LGB  YOLOv8

In [26]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
N_total      = 35577
C            = 15

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [15]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolov8n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SIMD_LGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SIMD_LGB_YOLOv8n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_LGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SIMD_LGB_YOLOv8n_weighted, nbs=64, nms=None, opse

In [27]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_LGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_LGB_YOLOv8n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_LGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_LGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON LGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB VALIDATION RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON LGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB TEST RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('LGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("LGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")

print("\n✓ Results saved to 'LGB_COMPLETE_results_nc15.txt'")
print("\n" + "="*50)
print("LGB COMPLETE!")
print("="*50)

Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_LGB\test\labels.cache
✓ Created: SIMD_LGB_val_nc15.yaml
✓ Created: SIMD_LGB_test_nc15.yaml

RUNNING VALIDATION ON LGB
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 509.5468.0 MB/s, size: 242.7 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels... 759 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 2.1Kit/s 0.4s0.1ss
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 4.6it/s 10.4s0.2s
                   all        759       6720      0.749       0.73      0.779      0.619
                   car        383       25

## RGB YOLOv12

In [28]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
N_total      = 35577
C            = 15

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [15]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo12n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SIMD_RGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SIMD_RGB_YOLOv12n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_RGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt

In [29]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_RGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_RGB_YOLO12n_epochs\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_RGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_RGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON RGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB VALIDATION RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON RGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB TEST RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('RGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("RGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_RGB\test\labels.cache
✓ Created: SIMD_RGB_val_nc15.yaml
✓ Created: SIMD_RGB_test_nc15.yaml

RUNNING VALIDATION ON RGB
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLOv12n summary (fused): 159 layers, 2,559,653 parameters, 0 gradients, 7.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1375.3307.5 MB/s, size: 242.9 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 2.3Kit/s 0.2s0.0ss
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 5.1it/s 6.3s0.2s
                   all        500       3218      0.713      0.674      0.722      0.567
                   car        230     

## YGB YOLOv12

In [30]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
N_total      = 35577
C            = 15

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [15]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo12n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SIMD_YGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SIMD_YGB_YOLOv12n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_YGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt

In [31]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_YGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_YGB_YOLO12n_epochs\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_YGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_YGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON YGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB VALIDATION RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON YGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB TEST RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('YGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("YGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_YGB\test\labels.cache
✓ Created: SIMD_YGB_val_nc15.yaml
✓ Created: SIMD_YGB_test_nc15.yaml

RUNNING VALIDATION ON YGB
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLOv12n summary (fused): 159 layers, 2,559,653 parameters, 0 gradients, 7.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1304.0309.2 MB/s, size: 241.1 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels... 759 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 2.1Kit/s 0.4s0.1ss
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 5.1it/s 9.4s0.2s
                   all        759       6720        0.7      0.709       0.75      0.582
                   car        383     

## LGB YOLOv12

In [32]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
N_total      = 35577
C            = 15

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [15]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo12n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SIMD_LGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SIMD_LGB_YOLOv12n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_LGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SIMD_LGB_YOLOv12n_weighted, nbs=64, nms=None, ops

In [33]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_LGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_LGB_YOLO12n_epochs\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_LGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_LGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON LGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB VALIDATION RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON LGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB TEST RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('LGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("LGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_LGB\test\labels.cache
✓ Created: SIMD_LGB_val_nc15.yaml
✓ Created: SIMD_LGB_test_nc15.yaml

RUNNING VALIDATION ON LGB
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLOv12n summary (fused): 159 layers, 2,559,653 parameters, 0 gradients, 7.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1775.6293.3 MB/s, size: 242.7 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels... 759 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 2.2Kit/s 0.3s0.1ss
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 5.0it/s 9.7s0.2s
                   all        759       6720      0.684      0.683      0.744      0.575
                   car        383     

## RGB  YOLOv26

In [34]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
N_total      = 35577
C            = 15

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [15]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo26n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SIMD_RGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SIMD_RGB_YOLO26n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_RGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SIMD_RGB_YOLO26n_weighted, nbs=64, nms=None, opse

In [35]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_RGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_RGB_YOLO26n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_RGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_RGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON RGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB VALIDATION RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON RGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("RGB TEST RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('RGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("RGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_RGB\test\labels.cache
✓ Created: SIMD_RGB_val_nc15.yaml
✓ Created: SIMD_RGB_test_nc15.yaml

RUNNING VALIDATION ON RGB
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLO26n summary (fused): 120 layers, 2,377,761 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1712.4426.9 MB/s, size: 242.9 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 2.4Kit/s 0.2s0.0ss
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 5.5it/s 5.8s0.2s
                   all        500       3218       0.67      0.737      0.755      0.591
                   car        230      

## YGB  YOLOv26

In [36]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
N_total      = 35577
C            = 15

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [15]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo26n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SIMD_YGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SIMD_YGB_YOLO26n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_YGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SIMD_YGB_YOLO26n_weighted, nbs=64, nms=None, opse

In [37]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_YGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_YGB_YOLO26n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_YGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_YGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON YGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB VALIDATION RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON YGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("YGB TEST RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('YGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("YGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_YGB\test\labels.cache
✓ Created: SIMD_YGB_val_nc15.yaml
✓ Created: SIMD_YGB_test_nc15.yaml

RUNNING VALIDATION ON YGB
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLO26n summary (fused): 120 layers, 2,377,761 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1495.9193.7 MB/s, size: 241.1 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels... 759 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 2.2Kit/s 0.3s0.0ss
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 5.5it/s 8.8s0.2s
                   all        759       6720      0.647      0.756       0.76      0.594
                   car        383      

## LGB  YOLOv26

In [38]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ── Same weight computation as above ──────────────────────────────────────────
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
N_total      = 35577
C            = 15

raw_weights  = [N_total / (C * n) for n in class_counts]
max_w        = max(raw_weights)                          # ≈ 57.85
norm_weights = [w / max_w for w in raw_weights]

weights_tensor = torch.tensor(norm_weights, dtype=torch.float32)  # shape [15]

# ── Custom Trainer — injects Eq. (8) weighted BCE into YOLO loss ──────────────
class WeightedDetectionTrainer(DetectionTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        device = next(model.parameters()).device
        w      = weights_tensor.to(device)              # move to GPU

        # Replace BCEcls with per-class weighted version — implements Eq. (8):
        # L_cls = -Σ w̃_i [ y_i log(ŷ_i) + (1-y_i) log(1-ŷ_i) ]
        if hasattr(model, 'criterion') and model.criterion is not None:
            model.criterion.BCEcls = nn.BCEWithLogitsLoss(
                pos_weight = w,
                reduction  = 'none'
            )
            print(f"✅ Weighted BCE injected | max_w={max_w:.2f} | "
                  f"fighter w̃={norm_weights[9]:.4f} | "
                  f"car w̃={norm_weights[0]:.4f}")
        return model


# ── Training call — same hyperparams as your paper ────────────────────────────
model   = YOLO('yolo26n.pt')    # or 'yolo26n.pt'
model.trainer = WeightedDetectionTrainer

results = model.train(
    data          = 'SIMD_LGB/data.yaml',
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,            # paper: batch=8
    lr0           = 0.1,          # paper: lr0=0.1
    optimizer     = 'SGD',
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    cos_lr        = True,         # paper: cosine annealing
    augment       = True,
    pretrained    = False,        # paper: trained from scratch
    patience      = 15,           # paper: early stopping patience=15
    device        = 0,
    project       = 'runs',
    name          = 'SIMD_LGB_YOLO26n_weighted',
    exist_ok      = True,
    seed          = 42,
    workers       = 4,
    amp           = False,
    resume        = False,
    cache         = False,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_LGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SIMD_LGB_YOLO26n_weighted, nbs=64, nms=None, opse

In [39]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_LGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_LGB_YOLO26n_weighted\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_LGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_LGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("\n" + "="*50)
print("RUNNING VALIDATION ON LGB")
print("="*50)
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB VALIDATION RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("\n" + "="*50)
print("PER-CLASS RESULTS (mAP50)")
print("="*50)
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("\n" + "="*50)
print("RUNNING TESTING ON LGB")
print("="*50)
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("\n" + "="*50)
print("LGB TEST RESULTS (nc=15)")
print("="*50)
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('LGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("LGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write("-" * 20 + "\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_LGB\test\labels.cache
✓ Created: SIMD_LGB_val_nc15.yaml
✓ Created: SIMD_LGB_test_nc15.yaml

RUNNING VALIDATION ON LGB
Ultralytics 8.4.160  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLO26n summary (fused): 120 layers, 2,377,761 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1754.1359.0 MB/s, size: 242.7 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels... 759 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 759/759 2.3Kit/s 0.3s0.0ss
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 5.6it/s 8.6s0.2s
                   all        759       6720      0.735      0.697      0.754      0.592
                   car        383      